In [1]:
import pandas as pd

# 1. Load Training Data
train_args = pd.read_csv("data/arguments-training.tsv", sep='\t')
train_labels = pd.read_csv("data/labels-training.tsv", sep='\t')
df_train = pd.merge(train_args, train_labels, on="Argument ID")

# 2. Load Validation Data
val_args = pd.read_csv("data/arguments-validation.tsv", sep='\t')
val_labels = pd.read_csv("data/labels-validation.tsv", sep='\t')
df_val = pd.merge(val_args, val_labels, on="Argument ID")

# 3. Load Test Data (Crucial for volume!)
test_args = pd.read_csv("data/arguments-test.tsv", sep='\t')
test_labels = pd.read_csv("data/labels-test.tsv", sep='\t')
df_test = pd.merge(test_args, test_labels, on="Argument ID")

# 4. Concatenate EVERYTHING into one giant dataset
trainval_df = pd.concat([df_train, df_val, df_test], ignore_index=True)

# 5. Verify the size (Should be > 8,500)
print(f"Total Examples: {len(trainval_df)}")
print(trainval_df.head(3))

Total Examples: 8865
  Argument ID                                   Conclusion       Stance  \
0      A01002                  We should ban human cloning  in favor of   
1      A01005                      We should ban fast food  in favor of   
2      A01006  We should end the use of economic sanctions      against   

                                             Premise  Self-direction: thought  \
0  we should ban human cloning as it will only ca...                        0   
1  fast food should be banned because it is reall...                        0   
2  sometimes economic sanctions are the only thin...                        0   

   Self-direction: action  Stimulation  Hedonism  Achievement  \
0                       0            0         0            0   
1                       0            0         0            0   
2                       0            0         0            0   

   Power: dominance  ...  Tradition  Conformity: rules  \
0                 0  ...          

In [2]:
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# 1. Create the Input Feature (X) and Targets (y) from your NEW trainval_df
# Concatenate Conclusion + Stance + Premise
trainval_df['text'] = trainval_df['Conclusion'] + " " + trainval_df['Stance'] + " " + trainval_df['Premise']

label_cols = [col for col in train_labels.columns if col != 'Argument ID']

# Create the arrays for splitting
X_all = trainval_df['text'].values
y_all = trainval_df[label_cols].values

print(f"Features shape: {X_all.shape}")
print(f"Labels shape:   {y_all.shape}")

# 2. Iterative Stratified Split (Train vs Test)
# We use X_all and y_all here
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# FIX: Use 'X_all' and 'y_all' inside the loop
for train_index, test_index in msss.split(X_all, y_all):
    X_train, X_test = X_all[train_index], X_all[test_index]
    y_train, y_test = y_all[train_index], y_all[test_index]

print("-" * 30)
print(f"Final Training Set: {X_train.shape[0]} examples (Use for Cross-Validation)")
print(f"Final Test Set:     {X_test.shape[0]} examples (Use for Report)")

# OPTIONAL: Sanity Check
print("\nLabel Distribution Check (First 3 labels):")
print(f"Train: {np.mean(y_train, axis=0)[:3]}")
print(f"Test:  {np.mean(y_test, axis=0)[:3]}")

Features shape: (8865,)
Labels shape:   (8865, 20)
------------------------------
Final Training Set: 7112 examples (Use for Cross-Validation)
Final Test Set:     1753 examples (Use for Report)

Label Distribution Check (First 3 labels):
Train: [0.15551181 0.25674916 0.05202475]
Test:  [0.15744438 0.2601255  0.05248146]


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.metrics import f1_score
import time

# Configuración de Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# SEMILLA (CRÍTICO para reproducibilidad)
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# Función de Entrenamiento Genérica (Sirve para LSTM y BERT)
def train_engine(model, train_loader, val_loader, epochs=5, lr=1e-3, is_bert=False):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss() # Obligatorio para Multi-label
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_f1': []}
    
    print(f"Iniciando entrenamiento ({'BERT' if is_bert else 'LSTM'})...")
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            if is_bert:
                input_ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(input_ids, mask)
            else:
                inputs, labels = batch
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Evaluación
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                if is_bert:
                    input_ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    labels = batch['labels'].to(device)
                    outputs = model(input_ids, mask)
                else:
                    inputs, labels = batch
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                
                # Sigmoid + Umbral 0.5 para decisión multietiqueta
                preds = torch.sigmoid(outputs) > 0.5
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        history['train_loss'].append(train_loss/len(train_loader))
        history['val_f1'].append(val_f1)
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f}")
        
    return history, model

Usando dispositivo: cuda


In [4]:
from collections import Counter

# --- 2.1 Preprocesamiento LSTM (Vocabulario) ---
class Vocabulary:
    def __init__(self, texts, max_size=10000):
        word_counts = Counter()
        for text in texts:
            word_counts.update(text.lower().split())
        
        self.vocab = {"<PAD>": 0, "<UNK>": 1}
        for word, _ in word_counts.most_common(max_size):
            self.vocab[word] = len(self.vocab)
            
    def text_to_indices(self, text, max_len=100):
        tokens = text.lower().split()
        indices = [self.vocab.get(t, 1) for t in tokens[:max_len]]
        indices += [0] * (max_len - len(indices)) # Padding
        return torch.tensor(indices, dtype=torch.long)

# Crear vocabulario con los datos de entrenamiento
vocab_handler = Vocabulary(X_train)

# Dataset Class
class LSTMDataset(Dataset):
    def __init__(self, texts, labels, vocab_handler):
        self.texts = texts
        self.labels = labels
        self.vh = vocab_handler
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        return self.vh.text_to_indices(self.texts[idx]), torch.tensor(self.labels[idx], dtype=torch.float)

# DataLoaders
train_ds_lstm = LSTMDataset(X_train, y_train, vocab_handler)
test_ds_lstm = LSTMDataset(X_test, y_test, vocab_handler)
train_loader_lstm = DataLoader(train_ds_lstm, batch_size=64, shuffle=True)
test_loader_lstm = DataLoader(test_ds_lstm, batch_size=64)

# --- 2.2 Arquitectura del Modelo LSTM ---
class HumanValueLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, n_classes=20):
        super().__init__()
        # 1. SEQUENCE ENCODER
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        
        # 2. CLASSIFICATION HEAD
        self.fc = nn.Linear(hidden_dim * 2, n_classes) # *2 por bidireccional
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        
        # 3. POOLING (Global Max Pooling)
        # Tomamos el valor máximo a través de la secuencia para capturar la característica más fuerte
        pooled, _ = torch.max(lstm_out, dim=1) 
        
        return self.fc(self.dropout(pooled))

# --- 2.3 Ejecutar LSTM ---
lstm_model = HumanValueLSTM(len(vocab_handler.vocab), n_classes=y_train.shape[1])
history_lstm, trained_lstm = train_engine(lstm_model, train_loader_lstm, test_loader_lstm, epochs=10, lr=0.001)

Iniciando entrenamiento (LSTM)...
Epoch 1/10 | Loss: 0.4324 | Val F1: 0.0000
Epoch 2/10 | Loss: 0.3970 | Val F1: 0.0637
Epoch 3/10 | Loss: 0.3753 | Val F1: 0.1326
Epoch 4/10 | Loss: 0.3556 | Val F1: 0.1820
Epoch 5/10 | Loss: 0.3414 | Val F1: 0.2255
Epoch 6/10 | Loss: 0.3290 | Val F1: 0.2477
Epoch 7/10 | Loss: 0.3195 | Val F1: 0.2837
Epoch 8/10 | Loss: 0.3096 | Val F1: 0.2888
Epoch 9/10 | Loss: 0.3017 | Val F1: 0.2992
Epoch 10/10 | Loss: 0.2932 | Val F1: 0.3061
